# 06. Classification

> Train and evaluate baseline classifiers (LDA, SVM, logistic regression) on the multi-band-power features using stratified k-fold cross-validation.

Reports accuracy, confusion matrices, and per-class metrics for the 3-class gesture paradigm (fist / peace / open). Each classifier is wrapped behind a uniform interface so it can be plugged into the benchmark in `09_benchmark`.

In [ ]:
#| default_exp classification

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Classifier zoo

Each entry is a *factory* (no-arg callable returning a fresh estimator) so the cross-validation loop can build a clean instance per fold without leaking state.

In [ ]:
#| export
CLASSIFIERS = {
    'LDA':      lambda: LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'),
    'SVM-rbf':  lambda: SVC(kernel='rbf', C=1.0, gamma='scale'),
    'logreg':   lambda: LogisticRegression(max_iter=2000, C=1.0),
}

## Pipeline + cross-validation

Standardization is fit on each training fold (no leakage) and applied to the matching test fold. Stratified 5-fold preserves class balance — with 30 trials per class, each fold has 6 test trials per class.

In [ ]:
#| export
def make_pipeline(clf):
    """Standardize, then classify."""
    return Pipeline([('scale', StandardScaler()), ('clf', clf)])

def cross_validate(make_clf, X, y, n_splits=5, random_state=0):
    """Stratified k-fold accuracy + summed confusion matrix.

    `make_clf` is a no-arg callable returning a fresh classifier per fold.
    Returns `(per_fold_accuracy, summed_confusion_matrix, class_labels)`."""
    classes = np.unique(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accs, cm_total = [], np.zeros((len(classes), len(classes)), dtype=int)
    for tr, te in skf.split(X, y):
        clf = make_pipeline(make_clf())
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        accs.append(accuracy_score(y[te], pred))
        cm_total += confusion_matrix(y[te], pred, labels=classes)
    return np.array(accs), cm_total, classes

## Run the bake-off

In [ ]:
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
from br41n_ecog_hand_pose.data import load_ecog, GESTURE_NAMES
from br41n_ecog_hand_pose.preprocessing import preprocess
from br41n_ecog_hand_pose.epoching import epoch_recording
from br41n_ecog_hand_pose.features import multi_band_power

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

In [ ]:
#| eval: false
rec = load_ecog()
clean, _ = preprocess(rec.ecog, rec.fs)
epochs, classes = epoch_recording(rec, tmin=0.0, tmax=2.0, signal=clean)
X = multi_band_power(epochs, rec.fs)
y = classes
print(f'X={X.shape}, y={y.shape}, chance = {1/len(np.unique(y)):.3f}')

In [ ]:
#| eval: false
results = {}
for name, make_clf in CLASSIFIERS.items():
    accs, cm, labels = cross_validate(make_clf, X, y)
    results[name] = (accs, cm, labels)
    print(f'{name:>8}: {accs.mean():.3f} \u00b1 {accs.std():.3f}   (folds: {[f"{a:.2f}" for a in accs]})')

In [ ]:
#| eval: false
fig, axs = plt.subplots(1, len(results), figsize=(4 * len(results), 3.5))
for ax, (name, (accs, cm, labels)) in zip(axs, results.items()):
    norm = cm / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels([GESTURE_NAMES[c] for c in labels])
    ax.set_yticklabels([GESTURE_NAMES[c] for c in labels])
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{name} — {accs.mean():.3f}')
    ax.grid(False)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f'{norm[i, j]:.2f}', ha='center', va='center',
                    color='white' if norm[i, j] > 0.5 else 'black', fontsize=9)
fig.suptitle('Confusion matrices (row-normalized)')
plt.tight_layout(); plt.show()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()